## Bidirectional LSTM Model

Bidirectional LSTMs process text sequences in both forward and backward directions.  
This allows the model to incorporate information from both past and future words when predicting emotions.  
Such bidirectional context is especially beneficial for short texts like tweets, where meaning often depends on surrounding words.

In [1]:
import pandas as pd
import numpy as np
import random
import os
import pickle

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [2]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


In [3]:
BASE_PATH = "../dataset"

train_df = pd.read_csv(os.path.join(BASE_PATH, "train_clean.csv"))
test_df = pd.read_csv(os.path.join(BASE_PATH, "test_clean.csv"))

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


Train shape: (16000, 2)
Test shape: (2000, 2)


,clean_text,label
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["label"])
y_test = label_encoder.transform(test_df["label"])

num_classes = len(label_encoder.classes_)
print("Classes:", label_encoder.classes_)


Classes: ['anger' 'fear' 'joy' 'love' 'sadness' 'surprise']


In [5]:
VOCAB_SIZE = 20000
MAX_LEN = 50

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(train_df["clean_text"])

X_train_seq = tokenizer.texts_to_sequences(train_df["clean_text"])
X_test_seq = tokenizer.texts_to_sequences(test_df["clean_text"])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post")

print("Padded train shape:", X_train_pad.shape)


Padded train shape: (16000, 50)


In [6]:
model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        input_length=MAX_LEN
    ),
    Bidirectional(LSTM(128, return_sequences=False)),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 50, 128)           2560000   
                                                                 
 bidirectional (Bidirection  (None, 256)               263168    
 al)                                                             
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense (Dense)               (None, 6)                 1542      
                                                                 
Total params: 2824710 (10.78 MB)
Trainable params: 2824710 (10.78 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [7]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/10
225/225 [==============================] - 16s 67ms/step - loss: 1.2977 - accuracy: 0.5015 - val_loss: 0.7331 - val_accuracy: 0.7269
Epoch 2/10
225/225 [==============================] - 15s 68ms/step - loss: 0.4543 - accuracy: 0.8495 - val_loss: 0.3345 - val_accuracy: 0.8881
Epoch 3/10
225/225 [==============================] - 15s 67ms/step - loss: 0.1900 - accuracy: 0.9387 - val_loss: 0.3498 - val_accuracy: 0.8869
Epoch 4/10
225/225 [==============================] - 15s 69ms/step - loss: 0.1167 - accuracy: 0.9615 - val_loss: 0.3980 - val_accuracy: 0.8906


In [8]:
y_pred_prob = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1-score (weighted): {f1:.4f}")


63/63 [==============================] - 1s 12ms/step
Test Accuracy: 0.8810
Test F1-score (weighted): 0.8716


In [9]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)


              precision    recall  f1-score   support

       anger       0.82      0.90      0.86       275
        fear       0.82      0.87      0.84       224
         joy       0.91      0.93      0.92       695
        love       0.86      0.71      0.78       159
     sadness       0.92      0.94      0.93       581
    surprise       0.64      0.14      0.23        66

    accuracy                           0.88      2000
   macro avg       0.83      0.75      0.76      2000
weighted avg       0.88      0.88      0.87      2000



### BiLSTM Model Observations

- The BiLSTM model captures context from both directions
- Performance is slightly improved compared to the unidirectional LSTM
- Computational cost is higher, but manageable for short texts
- Some rare emotion classes remain challenging


In [10]:
MODEL_PATH = "../models"
os.makedirs(MODEL_PATH, exist_ok=True)

model.save(os.path.join(MODEL_PATH, "bilstm_emotion_model.keras"))

with open(os.path.join(MODEL_PATH, "bilstm_tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(MODEL_PATH, "bilstm_label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)

print("BiLSTM model and artifacts saved.")


BiLSTM model and artifacts saved.


In [11]:
assert os.path.exists(os.path.join(MODEL_PATH, "bilstm_emotion_model.keras"))
assert os.path.exists(os.path.join(MODEL_PATH, "bilstm_tokenizer.pkl"))

print("Notebook 4 verified successfully.")


Notebook 4 verified successfully.


### Conclusion

The Bidirectional LSTM model further improves performance compared to the
unidirectional LSTM by capturing contextual information from both past
and future words in a sentence. This is particularly effective for short
texts such as tweets, where emotion can depend on surrounding context.

However, the improvement over the LSTM remains incremental, while the
computational cost increases. Some minority emotion classes (e.g.,
*surprise*) remain challenging due to class imbalance, indicating that
model architecture alone is not sufficient to address data imbalance.

This motivates the use of Transformer-based models, which leverage
self-attention and contextual embeddings to better capture global
dependencies in text.
